In [ ]:
# Fine tuning a BERT model with Huggingface pipelines.
# Retrains a base model with a new dataset, and saves the fine-tuned model
# Loads that new model and performs inference on novel data
# RTS, April 18, 2024.
#------------------------------------------------

# Canonical paper on  Large Language Model Transformer model (BERT):
# Jacob Devlin et al. BERT: Pre-training of Deep Bidirectional Transformers for Language Understanding:
# https://arxiv.org/abs/1810.04805

# Huggingface tutorial on BERT:
# https://huggingface.co/blog/bert-101

# Training dataset
# https://huggingface.co/datasets/yelp_review_full

# detailed
# https://github.com/NielsRogge/Transformers-Tutorials/blob/master/Mistral/Supervised_fine_tuning_(SFT)_of_an_LLM_using_Hugging_Face_tooling.ipynb

# pipelines outlined
# https://huggingface.co/docs/transformers/pipeline_tutorial

# model use with pipelines
# https://huggingface.co/docs/trl/main/en/use_model

In [ ]:
# Use a GPU runtime to run this notebook (change runtime type)
!nvidia-smi

Tue Apr 23 20:29:02 2024       
+---------------------------------------------------------------------------------------+
| NVIDIA-SMI 535.104.05             Driver Version: 535.104.05   CUDA Version: 12.2     |
|-----------------------------------------+----------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id        Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |         Memory-Usage | GPU-Util  Compute M. |
|                                         |                      |               MIG M. |
|=========================================+======================+======================|
|   0  Tesla T4                       Off | 00000000:00:04.0 Off |                    0 |
| N/A   34C    P8               9W /  70W |      0MiB / 15360MiB |      0%      Default |
|                                         |                      |                  N/A |
+-----------------------------------------+----------------------+--

In [ ]:
! pip install -U accelerate
! pip install -U transformers
! pip install -q transformers datasets
! pip install evaluate

In [ ]:
import os, sys
from google.colab import drive
drive.mount('/content/drive')
#change this based on your setup
root = '/content/drive/MyDrive/ART/machinelearning/'
sys.path.append(root +'code/')
datapath =  root + 'data/'

Mounted at /content/drive


In [ ]:
from datasets import load_dataset

# https://huggingface.co/datasets/yelp_review_full
dataset = load_dataset("yelp_review_full")
dataset["train"][100]


In [ ]:
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained("google-bert/bert-base-cased")


tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/213k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/436k [00:00<?, ?B/s]

In [ ]:
def tokenize_function(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True)

In [ ]:
# This step takes 25 minutes on CPU; 10 minutes on GPU (T4)
tokenized_datasets = dataset.map(tokenize_function, batched=True)

Map:   0%|          | 0/650000 [00:00<?, ? examples/s]

Map:   0%|          | 0/50000 [00:00<?, ? examples/s]

In [ ]:
# Use only a small training set
small_train_dataset = tokenized_datasets["train"].shuffle(seed=42).select(range(1000))
small_eval_dataset = tokenized_datasets["test"].shuffle(seed=42).select(range(1000))

In [ ]:
from transformers import AutoModelForSequenceClassification
#The BERT model was pretrained on BookCorpus, a dataset consisting of 11,038 unpublished books and English Wikipedia (excluding lists, tables and headers).
model = AutoModelForSequenceClassification.from_pretrained("google-bert/bert-base-cased", num_labels=5)

model.safetensors:   0%|          | 0.00/436M [00:00<?, ?B/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at google-bert/bert-base-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [ ]:
from transformers import TrainingArguments
training_args = TrainingArguments(output_dir = datapath + "huggingface_finetune")

In [ ]:
import numpy as np
import evaluate

metric = evaluate.load("accuracy")

In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return metric.compute(predictions=predictions, references=labels)

In [ ]:
from transformers import TrainingArguments, Trainer

# Defaults apply to parameters not explicitly set
training_args = TrainingArguments(
    output_dir = datapath + "huggingface_finetune",
    learning_rate=2e-5,
    num_train_epochs=5,
    weight_decay=0.01,
    evaluation_strategy="epoch")

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=small_train_dataset,
    eval_dataset=small_eval_dataset,
    compute_metrics=compute_metrics,
)

In [ ]:
#Now fine-tune; 5 epochs in 10 minutes on GPU
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy
1,No log,1.145319,0.541000
2,No log,1.029444,0.561000
3,No log,1.005410,0.590000


Epoch,Training Loss,Validation Loss,Accuracy
1,No log,1.145319,0.541000
2,No log,1.029444,0.561000
3,No log,1.005410,0.590000
4,0.950700,1.021962,0.591000
5,0.950700,1.042379,0.595000


TrainOutput(global_step=625, training_loss=0.8400389221191407, metrics={'train_runtime': 670.9511, 'train_samples_per_second': 7.452, 'train_steps_per_second': 0.932, 'total_flos': 1315590712320000.0, 'train_loss': 0.8400389221191407, 'epoch': 5.0})

In [ ]:
# Save the model
# https://discuss.huggingface.co/t/saving-a-model-and-loading-it/21492
# https://discuss.huggingface.co/t/loading-and-saving-a-model/74870

pt_save_directory = datapath + "huggingface_finetune"
tokenizer.save_pretrained(pt_save_directory)
model.save_pretrained(pt_save_directory)

In [ ]:
# Now do inference on the saved model with new input
from transformers import BertTokenizer, TFBertModel, TFBartForConditionalGeneration
finetunedmodel = datapath + "huggingface_finetune"

import tensorflow as tf
sample = "This restaurant is the most atrocious miserable place I have ever visited."

tokenizer = BertTokenizer.from_pretrained(finetunedmodel)
model = TFBertModel.from_pretrained(finetunedmodel)
encoded_input = tokenizer(sample, return_tensors='tf')
output = model(encoded_input)
print(output)
# so now decode that output !

In [ ]:
from transformers import pipeline
finetunedmodel = datapath + "huggingface_finetune"

#task = "text-classification"
#task = "text-generation"
task = "sentiment-analysis"

sample = "This restaurant is the most fabulous place I have ever visited."
sample2 = "This restaurant is the most atrocious, gross, miserable "

pipe = pipeline(task, model=finetunedmodel)

result = pipe(sample)
result2 = pipe(sample2)

In [ ]:
if(task == "text-generation"):
    print(result2[0]["generated_text"])
elif(task == "sentiment-analysis"):
    print(result[0]['label'], result[0]['score'])

5 stars 0.7909491062164307


In [ ]:
# The yelp review dataset has 5 class of reviews from 1 to 5, with 5 the highest
# the config.json model file has the generic label names LABEL_0 ...LABEL_4)
# I changed the config file to match the database labels
print(model.config.label2id)


{'1 star': 0, '2 stars': 1, '3 stars': 2, '4 stars': 3, '5 stars': 4}
